In [21]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib
from tqdm.notebook import tqdm

In [22]:
torch.manual_seed(11037)

In [23]:
class ProjectRedMLP(nn.Module):
    
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

        self.hidden = nn.Sequential(    nn.Linear(6, 32),
                                        nn.ReLU(),
                                        nn.Linear(32, 16),
                                        nn.ReLU()         )

        self.center = nn.Linear(16, 1)

        self.width = nn.Linear(16, 1)
        
    def forward(self, x):

        x = self.hidden(x)
        center = self.center(x)
        width = self.width(x) 

        return torch.cat((center, width), dim=1)

Softplus is because it is an activation that is linear at higher values so it doesn't blow up with vals.

In [24]:
model = ProjectRedMLP()
print(model)

ProjectRedMLP(
  (hidden): Sequential(
    (0): Linear(in_features=6, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=16, bias=True)
    (3): ReLU()
  )
  (center): Linear(in_features=16, out_features=1, bias=True)
  (width): Linear(in_features=16, out_features=1, bias=True)
)


In [25]:
df = pd.read_csv("project_red_v5_data_set.csv")
df.head()

,user_id,last_problem_id,user_rating,last_problem_rating,last_problem_solve_time_minutes,last_solve_time_outlier,mean_solve_time_minutes,time_std_minutes,problems_solved,center,width,lower_bound,upper_bound,synthetic
0,SYN_000000,2109:C3,2263.0,2600,48.62,typical,31.77,4.34,64,2412.24,302.23,2261.13,2563.36,True
1,SYN_000001,606:B,1414.0,1600,39.13,typical,31.28,9.94,84,1553.04,571.47,1267.30,1838.77,True
2,SYN_000002,913:D,1833.0,1800,37.25,typical,29.39,11.96,79,1825.24,444.26,1603.11,2047.37,True
3,SYN_000003,1419:D2,1290.0,1500,57.66,typical,41.23,17.98,142,1337.10,602.87,1035.67,1638.54,True
4,SYN_000004,1428:H,3385.0,3500,35.06,typical,31.50,17.31,306,3392.25,228.74,3277.88,3506.62,True


In [26]:
features = ["user_rating", "last_problem_rating", "mean_solve_time_minutes", "time_std_minutes", "problems_solved", "last_problem_solve_time_minutes"]

targets = ["center", "width"]

X = df[features]
Y = df[targets]

In [27]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=11037)

In [28]:
scaler_x = StandardScaler()

X_train = scaler_x.fit_transform(X_train)
X_test =  scaler_x.fit_transform(X_test)

In [29]:
joblib.dump(scaler_x, "scaler_x_v5.pkl")

['scaler_x_v5.pkl']

In [30]:
scaler_y = StandardScaler()

Y_train = scaler_y.fit_transform(Y_train)
Y_test =  scaler_y.fit_transform(Y_test)

In [31]:
X_train.dtype

dtype('float64')

In [32]:
Y_train.dtype

dtype('float64')

In [33]:
joblib.dump(scaler_y, "scaler_y_v5.pkl")

['scaler_y_v5.pkl']

In [34]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:",device)
if device.type == "cuda": print("GPU:", torch.cuda.get_device_name(0))

Using: cuda
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


In [35]:
model = model.to(device=device)

In [36]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
Y_train_tensor = torch.tensor(Y_train, dtype=torch.float32)
Y_test_tensor = torch.tensor(Y_test, dtype=torch.float32)

In [37]:
train_df = TensorDataset(X_train_tensor, Y_train_tensor)
test_df = TensorDataset(X_test_tensor, Y_test_tensor)

train_loader = DataLoader(train_df, batch_size=64, shuffle=True)
test_loader = DataLoader(test_df, batch_size=64, shuffle=False)

In [38]:
criterion = nn.MSELoss()

optim = torch.optim.Adam(model.parameters(), lr=0.001)

In [39]:
epochs = 100

train_losses = []
test_losses = []

progress = tqdm(range(epochs), desc="Training Gama", unit="epochs")

for epoch in progress:
    model.train()
    total_train_loss = 0

    for X_bch, Y_bch in train_loader:
        X_bch = X_bch.to(device)
        Y_bch = Y_bch.to(device)

        optim.zero_grad()

        preds = model(X_bch)

        loss = criterion(preds, Y_bch)

        loss.backward()

        optim.step()

        total_train_loss += loss.item() * X_bch.size(0)

    avg_train_loss = total_train_loss / len(train_df)
    train_losses.append(avg_train_loss)


    model.eval()
    total_test_loss = 0

    with torch.no_grad():
        for X_bch, Y_bch in test_loader:

            X_bch = X_bch.to(device)
            Y_bch = Y_bch.to(device)

            preds = model(X_bch)

            loss = criterion(preds, Y_bch)

            total_test_loss += loss.item() * X_bch.size(0)
    avg_test_loss = total_test_loss/len(test_df)
    test_losses.append(avg_test_loss)

    progress.set_postfix({"Train loss":f"{avg_train_loss}", "Test loss":f"{avg_test_loss}"})

Training Gama:   0%|          | 0/100 [00:00<?, ?epochs/s]

In [40]:
train_losses[:10]

[0.5381744019389153,
 0.07339447808265687,
 0.056256423592567446,
 0.05017093862593174,
 0.04715927977859974,
 0.0451020672917366,
 0.04412761023640633,
 0.04340449821949005,
 0.04304328542947769,
 0.04252639451622963]

In [41]:
torch.save(model.state_dict(), "gama_v5.pth")

In [44]:
model.eval()
dummy = torch.randn(1, 6, device=device)

In [45]:
torch.onnx.export(model, dummy, "gama_v5.onnx", input_names=["input"], output_names=["c/w"], opset_version=18, dynamo=True)

[torch.onnx] Obtain model graph for `ProjectRedMLP([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ProjectRedMLP([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/nix/store/wzif7pg2cmy12czmfgydd7xw919dy8z5-python3-3.13.15-env/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅


ONNXProgram(
    model=
        <
            ir_version=10,
            opset_imports={'': 18},
            producer_name='pytorch',
            producer_version='2.11.0',
            domain=None,
            model_version=None,
        >
        graph(
            name=main_graph,
            inputs=(
                %"input"<FLOAT,[1,6]>
            ),
            outputs=(
                %"c/w"<FLOAT,[1,2]>
            ),
            initializers=(
                %"hidden.0.weight"<FLOAT,[32,6]>{TorchTensor(...)},
                %"hidden.0.bias"<FLOAT,[32]>{TorchTensor(...)},
                %"hidden.2.weight"<FLOAT,[16,32]>{TorchTensor(...)},
                %"hidden.2.bias"<FLOAT,[16]>{TorchTensor(...)},
                %"center.weight"<FLOAT,[1,16]>{TorchTensor(...)},
                %"center.bias"<FLOAT,[1]>{TorchTensor<FLOAT,[1]>(Parameter containing: tensor([0.1757], device='cuda:0', requires_grad=True), name='center.bias')},
                %"width.weight"<FLOAT,[1,16]>{T

In [46]:
model.eval()

with torch.no_grad():
    prd = model(X_test_tensor.to(device)).cpu()

    center_MeanSquareError = ((prd[:,0] - Y_test_tensor[:,0]) ** 2).mean()
    width_MeanSquareError = ((prd[:,1] - Y_test_tensor[:,1]) ** 2).mean()

print("Center MSE:", center_MeanSquareError.item())
print("Width MSE:", width_MeanSquareError.item())
    

Center MSE: 0.008385083638131618
Width MSE: 0.08039167523384094
